In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
import gc
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. MEMORY OPTIMIZATION
# ==========================================
def optimize_dtypes(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type in ['object', 'string']:
            df[col] = df[col].astype('category')
        elif col_type == 'float64':
            df[col] = df[col].astype('float32')
        elif col_type == 'int64':
            df[col] = df[col].astype('int32')
    return df

# ==========================================
# 1. CONFIGURATION
# ==========================================
train_path = "/kaggle/input/competitions/ts-forecasting/train.parquet"
test_path = "/kaggle/input/competitions/ts-forecasting/test.parquet"
TARGET = "y_target"
FORECAST_WINDOWS = [1, 3, 10, 25]
N_SPLITS = 8 # Reduced to 5 for time-series splits to ensure decent training sizes

CORE_FEATURES = ['feature_al', 'feature_am', 'feature_cg', 'feature_by']

# Base Learner Config (Adding heavier regularization to combat chronological overfitting)
base_cfg = {
    'objective': 'regression',
    'metric': 'rmse',
    'n_estimators': 3000,
    'learning_rate': 0.015,
    'num_leaves': 63,          # Reduced slightly for robustness
    'max_depth': 7,            # Shallower trees
    'min_child_samples': 200,  # Require more samples per leaf
    'feature_fraction': 0.70, 
    'subsample': 0.80,        
    'verbosity': -1,
    'n_jobs': -1,
    'random_state': 42
}

# ==========================================
# 2. METRIC
# ==========================================
def weighted_rmse_score(y_true, y_pred, w):
    if w is None: w = np.ones_like(y_true)
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w * (y_true ** 2)) + 1e-8
    return float(np.sqrt(num / den))

# ==========================================
# 3. THE PIPELINE (STRICT CHRONOLOGICAL)
# ==========================================
test_outputs = []
overall_oof_y = []
overall_oof_pred = []
overall_oof_weights = []
importance_records = [] 

for horizon in FORECAST_WINDOWS:
    print(f"\n{'='*80}")
    print(f" HORIZON: {horizon} | TimeSeriesSplit Base -> Ridge Meta-Learner")
    print(f"{'='*80}")
    
    # Lazy Load Data
    tr_df = pd.read_parquet(train_path, engine='pyarrow', filters=[[('horizon', '==', horizon)]])
    te_df = pd.read_parquet(test_path, engine='pyarrow', filters=[[('horizon', '==', horizon)]])

    if len(tr_df) == 0 or len(te_df) == 0: continue

    # Ensure chronological sorting is absolute
    tr_df = tr_df.sort_values('ts_index').reset_index(drop=True)
    te_df = te_df.sort_values('ts_index').reset_index(drop=True)

    # Day of Week & Categoricals
    tr_df['day_of_week'] = (tr_df['ts_index'] % 7).astype('int32')
    te_df['day_of_week'] = (te_df['ts_index'] % 7).astype('int32')
    cat_cols = [c for c in tr_df.columns if tr_df[c].dtype in ['object', 'category', 'string']]
    
    # Safe Lags on Core Features
    cols_to_lag = [c for c in CORE_FEATURES if c in tr_df.columns]
    tr_df['is_test'], te_df['is_test'] = 0, 1
    combined = pd.concat([tr_df, te_df], ignore_index=True)
    
    group_obj = combined.groupby(cat_cols) if cat_cols else combined
    for col in cols_to_lag:
        combined[f'{col}_lag1'] = group_obj[col].shift(1).astype('float32')
        combined[f'{col}_lag2'] = group_obj[col].shift(2).astype('float32')

    tr_df = combined[combined['is_test'] == 0].drop(columns=['is_test']).reset_index(drop=True)
    te_df = combined[combined['is_test'] == 1].drop(columns=['is_test']).reset_index(drop=True)
    del combined, group_obj; gc.collect()

    # Prepare Types & Targets
    tr_df, te_df = optimize_dtypes(tr_df), optimize_dtypes(te_df)
    y_train = tr_df.pop(TARGET).astype(np.float32)
    weights = tr_df.pop('weight').astype(np.float32) if 'weight' in tr_df.columns else None

    drop_cols = ['ts_index', 'horizon', 'id']
    features = sorted([c for c in tr_df.columns if c in te_df.columns and c not in drop_cols])

    # Feature Dropout Array for Base Learners
    feature_contri_base = []
    for col in features:
        is_core = (col in CORE_FEATURES) or (col == 'day_of_week') or any(col.startswith(c + "_lag") for c in CORE_FEATURES)
        feature_contri_base.append(1.0 if is_core else 0.05)
            
    base_lgb_cfg = base_cfg.copy()
    base_lgb_cfg['feature_contri'] = feature_contri_base

    oof_preds = np.zeros(len(tr_df), dtype=np.float32)
    test_preds_blend = np.zeros(len(te_df), dtype=np.float32)
    
    # STRICT TIME-BASED SPLIT
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    
    # --- LEVEL 1: BASE FOLDS ---
    print(f"-> Training {N_SPLITS} Chronological Base Folds (LightGBM)...")
    for fold, (trn_idx, val_idx) in enumerate(tscv.split(tr_df)):
        X_trn, y_trn = tr_df[features].iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = tr_df[features].iloc[val_idx], y_train.iloc[val_idx]
        w_trn = weights.iloc[trn_idx] if weights is not None else None
        w_val = weights.iloc[val_idx] if weights is not None else None

        model = lgb.LGBMRegressor(**base_lgb_cfg)
        model.fit(X_trn, y_trn, sample_weight=w_trn, eval_set=[(X_val, y_val)],
                  eval_sample_weight=[w_val] if w_val is not None else None,
                  callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
        
        oof_preds[val_idx] = model.predict(X_val).astype(np.float32)
        # Weighting test predictions equally across folds
        test_preds_blend += model.predict(te_df[features]).astype(np.float32) / N_SPLITS
        
        for f, imp in zip(features, model.feature_importances_):
            importance_records.append({'horizon': horizon, 'model_type': 'base_lgbm', 'feature': f, 'importance': imp})

        print(f"   Fold {fold+1:02d} Trees: {model.best_iteration_}")
        del model, X_trn, X_val; gc.collect()

    # NOTE: TimeSeriesSplit leaves the very first training chunk without OOF predictions (since it's only used to train fold 1).
    # We must mask out the initial zeros in oof_preds before training the meta-learner, otherwise Ridge learns garbage.
    valid_oof_mask = oof_preds != 0

    # --- LEVEL 2: RIDGE META-LEARNER ---
    print("-> Training Meta-Learner (Ridge) strictly on Valid OOF Base Predictions...")
    
    # Alpha tuned down slightly to allow Ridge to fit to the now-honest, noisier OOF predictions
    meta_model = Ridge(alpha=1.0, fit_intercept=True) 
    
    X_meta_train = oof_preds[valid_oof_mask].reshape(-1, 1)
    y_meta_train = y_train.iloc[valid_oof_mask]
    w_meta_train = weights.iloc[valid_oof_mask] if weights is not None else None
    
    X_meta_test = test_preds_blend.reshape(-1, 1)

    meta_model.fit(X_meta_train, y_meta_train, sample_weight=w_meta_train)
    
    stacked_test_preds = meta_model.predict(X_meta_test).astype(np.float32)
    stacked_oof_preds = np.zeros_like(oof_preds)
    stacked_oof_preds[valid_oof_mask] = meta_model.predict(X_meta_train).astype(np.float32)

    importance_records.append({'horizon': horizon, 'model_type': 'meta_ridge', 'feature': 'base_pred_weight', 'importance': meta_model.coef_[0]})

    # Summary (Calculating score only on the valid OOF portion)
    horizon_w = weights.iloc[valid_oof_mask].values if weights is not None else np.ones_like(y_meta_train.values)
    base_score = weighted_rmse_score(y_meta_train.values, oof_preds[valid_oof_mask], horizon_w)
    stacked_score = weighted_rmse_score(y_meta_train.values, stacked_oof_preds[valid_oof_mask], horizon_w)
    
    print(f"\n>>> Horizon {horizon} Base Valid OOF WRMSE:    {base_score:.5f}")
    print(f">>> Horizon {horizon} Stacked Valid OOF WRMSE: {stacked_score:.5f}")
    
    overall_oof_y.append(y_meta_train.values)
    overall_oof_pred.append(stacked_oof_preds[valid_oof_mask])
    overall_oof_weights.append(horizon_w)

    te_ids = pd.read_parquet(test_path, engine='pyarrow', columns=['id'], filters=[[('horizon', '==', horizon)]])
    # IMPORTANT: Output column renamed to 'y_target' to prevent Kaggle from zeroing out missing columns
    te_ids[TARGET] = stacked_test_preds
    test_outputs.append(te_ids)

    del tr_df, te_df, X_meta_train, X_meta_test, meta_model; gc.collect()

# ==========================================
# 4. FINAL EVALUATION & EXPORT
# ==========================================
final_global_score = weighted_rmse_score(np.concatenate(overall_oof_y), np.concatenate(overall_oof_pred), np.concatenate(overall_oof_weights))
print(f"\n{'='*80}\n🏆 FINAL GLOBAL STACKED VALID OOF WRMSE: {final_global_score:.5f}\n{'='*80}\n")

# Generate Submission CSV
submission = pd.concat(test_outputs, ignore_index=True)
submission.dropna(subset=[TARGET], inplace=True)
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv successfully generated.")

# Process and Generate Feature Importances CSV
fi_df = pd.DataFrame(importance_records)
base_fi = fi_df[fi_df['model_type'] == 'base_lgbm'].groupby('feature')['importance'].mean().reset_index().rename(columns={'importance': 'avg_base_importance'})
base_fi = base_fi.sort_values('avg_base_importance', ascending=False)
meta_fi = fi_df[fi_df['model_type'] == 'meta_ridge'].groupby('feature')['importance'].mean().reset_index().rename(columns={'importance': 'ridge_coefficient'})

final_fi = pd.merge(base_fi, meta_fi, on='feature', how='outer').fillna(0.0)
final_fi.to_csv("feature_importances_summary.csv", index=False)
print("✅ feature_importances_summary.csv successfully generated.")